#Testing

In [3]:
MODEL_NAME = "Qwen/Qwen3.5-4B"
MAX_SEQ_LENGTH = 1536
DRIVE_ROOT = "/content/drive/MyDrive/datax_qwen35_4b_urdu_v5_language_safe"

In [3]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
try:
 import numpy, PIL
 _numpy = f"numpy=={numpy.__version__}"
 _pillow = f"pillow=={PIL.__version__}"
except Exception:
 _numpy, _pillow = "numpy", "pillow"
!uv pip install -qqq  "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pillow} torchvision bitsandbytes "xformers==0.0.32.post2"  "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo"  "unsloth[base] @ git+https://github.com/unslothai/unsloth"
!uv pip install -qqq --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" "trl==0.22.2" unsloth unsloth_zoo
!uv pip install -qqq "transformers==5.2.0" tensorboard rapidfuzz datasketch sentencepiece
!uv pip install -qqq --no-deps "torchcodec==0.7.0" "torchao>=0.16.0"
!uv pip install -qqq --no-build-isolation flash-linear-attention "causal_conv1d==1.6.0"
import torch
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
 !uv pip install -qqq --no-deps "apache-tvm-ffi==0.1.9" "tilelang==0.1.8"
else:
 os.environ["FLA_TILELANG"] = "0"

In [4]:
import os, sys, re, json
from pathlib import Path

import unsloth
import torch, transformers, trl, peft, bitsandbytes
from google.colab import drive, userdata

drive.mount("/content/drive")
HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "Add HF_TOKEN to Colab Secrets."

ROOT = Path(DRIVE_ROOT)
assert torch.cuda.is_available(), "A CUDA GPU is required"
bf16 = torch.cuda.is_bf16_supported()
MODEL_TORCH_DTYPE = torch.bfloat16 if bf16 else torch.float16
print({"Transformers": transformers.__version__, "TRL": trl.__version__, "model_dtype": str(MODEL_TORCH_DTYPE)})

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{'Transformers': '5.2.0', 'TRL': '0.22.2', 'model_dtype': 'torch.float16'}


In [5]:
from unsloth import FastLanguageModel

ADAPTER_PATH = ROOT / "best_adapter"
assert ADAPTER_PATH.exists(), f"Adapter not found: {ADAPTER_PATH}"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(ADAPTER_PATH),
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=MODEL_TORCH_DTYPE,
    load_in_4bit=True,
    token=HF_TOKEN,
)
FastLanguageModel.for_inference(model)

text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
if not getattr(text_tokenizer, "chat_template", None):
    text_tokenizer.chat_template = tokenizer.chat_template

IM_END_TOKEN = "<|im_end|>"
IM_END_ID = text_tokenizer.convert_tokens_to_ids(IM_END_TOKEN)
if text_tokenizer.pad_token_id is None:
    text_tokenizer.pad_token = IM_END_TOKEN

print("Trained adapter loaded successfully:", ADAPTER_PATH)

==((====))==  Unsloth 2026.7.3: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Trained adapter loaded successfully: /content/drive/MyDrive/datax_qwen35_4b_urdu_v5_language_safe/best_adapter


In [6]:
RUNTIME_SYSTEM_PROMPT = """
آپ احمد ہیں، ڈیٹا ایکس ٹیکنالوجیز کے ایک تجربہ کار سیلز کنسلٹنٹ کے طور پر ایک آؤٹ باؤنڈ کال پر گاہک سے بات کر رہے ہیں۔ یہ کال آپ نے خود شروع کی ہے، اس لیے سب سے پہلے اجازت لیں، پھر بات چیت آگے بڑھائیں۔ آپ کا مقصد گاہک کو ہماری خدمات کی افادیت پر قائل کرنا اور دلچسپی رکھنے والے گاہک کی رابطہ تفصیلات حاصل کرنا ہے۔

لازمی اصول:
۱۔ سب سے پہلے (پہلے جواب میں) اپنا اور کمپنی کا واضح تعارف دے چکے ہیں — اب پوچھیں کہ کیا گاہک کے پاس ابھی مختصر بات کے لیے وقت ہے، اس سے پہلے کہ کوئی خدمت پیش کریں۔
۲۔ اگر گاہک کہے کہ وقت نہیں ہے، مصروف ہے، یا دلچسپی نہیں رکھتا، تو ہرگز اصرار نہ کریں — فوراً شائستگی سے شکریہ ادا کریں اور کال مہذب انداز میں ختم کریں۔
۳۔ اگر گاہک بات کرنے پر راضی ہو تو مختصراً بتائیں کہ آپ کس مقصد کے لیے کال کر رہے ہیں، پھر ان کے کاروبار کی نوعیت اور موجودہ ضروریات کے بارے میں پوچھیں۔
۴۔ گفتگو کی پوری سابقہ معلومات یاد رکھیں۔ گاہک جو معلومات پہلے دے چکا ہو وہ دوبارہ نہ پوچھیں۔
۵۔ صرف انہی خدمات کا ذکر کریں جو منظور شدہ معلومات میں شامل ہیں: کسٹم سافٹ ویئر، ویب سائٹ ڈیزائن اور ڈویلپمنٹ، موبائل ایپلیکیشن، سی آر ایم سسٹمز، بزنس اور مینجمنٹ سسٹمز، بزنس آٹومیشن، ایس ای او، اور ڈیجیٹل مارکیٹنگ۔
۶۔ اعتماد اور فعال انداز میں گاہک کو ان خدمات کے فوائد سمجھائیں اور اگلے قدم کی طرف گفتگو بڑھائیں۔ کسی منصوبے، قیمت، رعایت، مدت، ضمانت یا کلائنٹ کا نام خود سے نہ بنائیں۔
۷۔ کبھی جھوٹی جلدی، دباؤ، یا ضمانت کا انداز استعمال نہ کریں۔ اعتماد اور ایمانداری کے ساتھ قائل کریں، دباؤ سے نہیں۔
۸۔ اگر گاہک قیمت یا مدت پوچھے تو بتائیں کہ یہ ضروریات کے جائزے کے بعد طے ہوتی ہے، اور فوراً ایک متعلقہ سوال پوچھ کر گفتگو کو دلچسپی کی طرف موڑیں۔
۹۔ جیسے ہی گاہک کی ضرورت واضح ہو جائے، صاف طور پر پوچھیں کہ کیا وہ چاہتا ہے کہ ٹیم اس سے تفصیل سے رابطہ کرے۔
۱۰۔ اگر گاہک دلچسپی ظاہر کرے تو بزنس کا نام، رابطہ نمبر، واٹس ایپ نمبر، اور ای میل ایک ایک کرکے پوچھیں، اور ہر معلومات ملنے کے بعد اسے واپس دہرا کر تصدیق کریں۔
۱۱۔ اگر گاہک واضح دلچسپی ظاہر نہ کرے، انکار کرے، یا دوبارہ کال نہ کرنے کو کہے، تو دوبارہ اصرار نہ کریں — شکریہ ادا کریں اور فوراً کال مہذب انداز میں ختم کریں۔
۱۲۔ "ٹیم" اور "کمپنی" جیسے مؤنث الفاظ کے ساتھ ہمیشہ درست صیغہ استعمال کریں، مثلاً "ٹیم رابطہ کرے گی"، کبھی "کرے گا" نہ کہیں۔
۱۳۔ جواب زیادہ سے زیادہ تین مختصر اور قدرتی اردو جملوں میں دیں۔
۱۴۔ ایک جواب میں صرف ایک سوال کریں۔
۱۵۔ انگریزی، رومن اردو، ستارے، سرخیاں اور فہرست استعمال نہ کریں۔
۱۶۔ اگر معلومات دستیاب نہ ہوں تو کہیں کہ متعلقہ ٹیم تصدیق کرکے بتائے گی۔
"""


In [7]:
import unicodedata
from transformers import LogitsProcessor

LEAK_PHRASES = ["سسٹم پرامپٹ", "اندرونی ہدایات", "تربیتی ڈیٹا"]

def is_arabic_script_letter(ch):
    cp = ord(ch)
    return ch.isalpha() and ((0x0600<=cp<=0x06FF) or (0x0750<=cp<=0x077F) or (0x0870<=cp<=0x089F) or (0x08A0<=cp<=0x08FF))

def has_foreign_letter(text): return any(ch.isalpha() and not is_arabic_script_letter(ch) for ch in text)

def token_piece_allowed(piece):
    if not piece or "\ufffd" in piece or any(ch in "<>{}" for ch in piece): return False
    if has_foreign_letter(piece): return False
    return not any(unicodedata.category(ch) in ("Cc","Cs") and not ch.isspace() for ch in piece)

# Build the mask against the model's REAL output vocab size, not tokenizer length
model_vocab_size = model.get_output_embeddings().weight.shape[0]
if model_vocab_size != len(text_tokenizer):
    print(f"Note: tokenizer length ({len(text_tokenizer)}) != model output vocab "
          f"({model_vocab_size}); clipping to the model's range.")

allowed_ids = []
for token_id in range(model_vocab_size):
    if token_id == IM_END_ID:
        allowed_ids.append(token_id); continue
    if token_id in text_tokenizer.all_special_ids: continue
    piece = text_tokenizer.decode([token_id], skip_special_tokens=False, clean_up_tokenization_spaces=False)
    if token_piece_allowed(piece): allowed_ids.append(token_id)
allowed_id_set = set(allowed_ids)
assert IM_END_ID in allowed_id_set and len(allowed_ids) > 100

class UrduOnlyLogitsProcessor(LogitsProcessor):
    def __init__(self, ids, vocab_size):
        mask = torch.zeros(vocab_size, dtype=torch.bool); mask[ids] = True
        self.allowed_cpu = mask; self.cache = {}
    def __call__(self, input_ids, scores):
        key = str(scores.device)
        if key not in self.cache: self.cache[key] = self.allowed_cpu.to(scores.device)
        return scores.masked_fill(~self.cache[key].unsqueeze(0), -float("inf"))

URDU_ONLY_PROCESSOR = UrduOnlyLogitsProcessor(allowed_ids, model_vocab_size)

# Raised from 96 — Urdu subword tokenization eats budget fast, and 96 tokens
# was cutting 2-3 sentence replies off mid-word/mid-sentence before the
# model ever reached IM_END_ID (visible in test transcripts as replies that
# just stop, e.g. "...رابطہ کر"). 200 gives real headroom; validate_output's
# 350-char cap still bounds the top end.
GENERATION_MAX_NEW_TOKENS = 200

def validate_output(t, finished=True):
    reasons = []
    if not finished: reasons.append("truncated")
    if has_foreign_letter(t) or "\ufffd" in t: reasons.append("foreign_script")
    if "<think>" in t.lower() or "</think>" in t.lower(): reasons.append("thinking")
    if any(x in t for x in LEAK_PHRASES): reasons.append("prompt_leak")
    if any(x in t for x in ("```","###","{","}")): reasons.append("format")
    if t.count("؟") > 1: reasons.append("questions")
    if len([x for x in re.split(r"[۔؟!?]+", t) if x.strip()]) > 3: reasons.append("sentences")
    if len(t) > 350 or not t.strip(): reasons.append("length")
    if re.search(r"(?:PKR|Rs\.?|\d+\s*روپے|سو فیصد|گارنٹی|پہلے نمبر)", t, re.I): reasons.append("unsupported")
    return reasons

def generate(messages, deterministic=True, max_new_tokens=GENERATION_MAX_NEW_TOKENS):
    prompt = text_tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", enable_thinking=False).to(model.device)
    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        repetition_penalty=1.08,
        logits_processor=[URDU_ONLY_PROCESSOR],
        eos_token_id=IM_END_ID,
        pad_token_id=text_tokenizer.pad_token_id,
        use_cache=True,
    )
    if deterministic:
        gen_kwargs["do_sample"] = False
    else:
        gen_kwargs.update(do_sample=True, temperature=0.7, top_p=0.9)
    with torch.inference_mode():
        out = model.generate(prompt, **gen_kwargs)
    gen_ids = out[0, prompt.shape[1]:]
    # False here means max_new_tokens ran out before the model produced
    # IM_END_ID itself — i.e. the reply was cut off mid-sentence, not that
    # the model chose to stop there.
    finished = IM_END_ID in gen_ids.tolist()
    text = text_tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    assert not has_foreign_letter(text), "Hard Urdu constraint failed"
    return text, finished

SAFE_FALLBACK = "معذرت، اس بارے میں صحیح معلومات ہماری ٹیم آپ کو دے گی۔"

def generate_agent_reply(conversation_history, verified_context=None):
    clean = []
    for m in conversation_history:
        if m.get("role") not in ("user","assistant"): continue
        text = re.sub(r"<think>.*?</think>", "", m.get("content",""), flags=re.S|re.I).strip()
        if m.get("role")=="assistant" and validate_output(text): continue
        if text: clean.append({"role": m["role"], "content": text})
    messages = [{"role":"system","content":RUNTIME_SYSTEM_PROMPT}] + clean
    if verified_context: messages.append({"role":"system","content":"صرف تصدیق شدہ معلومات: "+verified_context})

    # Pass 1: sampled. Pass 2: greedy. Both at the normal token budget.
    last_reasons = ["truncated"]
    for deterministic in (False, True):
        text, finished = generate(messages, deterministic=deterministic)
        reasons = validate_output(text, finished=finished)
        if not reasons:
            return text
        last_reasons = reasons

    # Extra-budget retry ONLY if truncation was the sole problem — more
    # tokens won't fix a foreign-script leak or a format violation, only
    # a reply that was legitimately cut off mid-sentence.
    if last_reasons == ["truncated"]:
        text, finished = generate(messages, deterministic=True, max_new_tokens=GENERATION_MAX_NEW_TOKENS + 80)
        if not validate_output(text, finished=finished):
            return text

    return SAFE_FALLBACK


Note: tokenizer length (248077) != model output vocab (248320); clipping to the model's range.


In [8]:
import torch
# FIX: Ensure URDU_ONLY_PROCESSOR matches the actual model output dimension to prevent RuntimeError
model_vocab_size = model.get_output_embeddings().weight.shape[0]
URDU_ONLY_PROCESSOR = UrduOnlyLogitsProcessor(allowed_ids, model_vocab_size)

In [9]:
!pip install -q pyngrok flask flask-cors

In [10]:
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
from google.colab import userdata
import threading

app = Flask(__name__)
CORS(app)

@app.route('/chat', methods=['POST'])
def chat_endpoint():
    data = request.json
    history = data.get('history', [])
    # history should be list of {'role': 'user', 'content': '...'}
    try:
        reply = generate_agent_reply(history)
        return jsonify({'reply': reply})
    except Exception as e:
        return jsonify({'error': str(e)}), 500

def run_flask():
    app.run(port=5000)

# Configure Ngrok
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Open a tunnel
public_url = ngrok.connect(5000).public_url
print(f" * Public URL for agent.py: {public_url}/chat")

# Start Flask in background
threading.Thread(target=run_flask).start()

 * Public URL for agent.py: https://astrology-stranger-capacity.ngrok-free.dev/chat


In [ ]:
import gradio as gr

def chat_with_model(message, history):
    conversation = []
    for item in history or []:
        if isinstance(item, dict):
            role, content = item.get("role"), item.get("content")
            if role in ("user","assistant") and isinstance(content, str):
                conversation.append({"role": role, "content": content.strip()})
        elif isinstance(item, (list, tuple)) and len(item) == 2:
            user_text, assistant_text = item
            if isinstance(user_text, str) and user_text.strip():
                conversation.append({"role":"user","content":user_text.strip()})
            if isinstance(assistant_text, str) and assistant_text.strip():
                conversation.append({"role":"assistant","content":assistant_text.strip()})
    conversation.append({"role":"user","content": str(message).strip()})
    try:
        return str(generate_agent_reply(conversation))
    except Exception as error:
        print("Generation error:", repr(error))
        return SAFE_FALLBACK

demo = gr.ChatInterface(
    fn=chat_with_model,
    title="Data X Urdu Assistant",
    description="اردو یا رومن اردو میں پیغام لکھیں",
    examples=["آپ کی کمپنی کیا خدمات فراہم کرتی ہے؟", "price ka idea bta dain", "میں ابھی مصروف ہوں"],
)
demo.launch(share=True, debug=True)

In [ ]:
print(generate_agent_reply([{"role":"user","content":"price ka idea bta dain"}]))
print(generate_agent_reply([{"role":"user","content":"آپ کی کمپنی کیا خدمات فراہم کرتی ہے؟"}]))

قیمت کاروبار کی مخصوص ضرورٹوں پر منحصر ہوتی ہے۔ کیا آپ مختصر طور پر اپنی ضرورت بتا سکتے ہیں تاکہ ٹیم مناسب رائے دے سکے؟
ضرور، ڈیٹا ایکس ٹیکنالوجیز کسٹم سافٹ ویئر ویب سائٹ ڈیزائن اور ڈویلپمنٹ موبائل ایپلیکیشن سی آر ایم سسٹمز بزنس آٹومیشن ایس ای او اور ڈیجیٹل مارکیٹنگ پر کام کرتی ہے۔ کیا آپ ابھی مختصر بات کے لیے وقت رکھ سکتے ہیں


In [ ]:
INBOUND_RUNTIME_SYSTEM_PROMPT = """
آپ احمد ہیں، ڈیٹا ایکس ٹیکنالوجیز کے ایک تجربہ کار سیلز کنسلٹنٹ کے طور پر گاہک سے بات کر رہے ہیں۔ آپ کا مقصد گاہک کو ہماری خدمات کی افادیت پر قائل کرنا اور دلچسپی رکھنے والے گاہک کی رابطہ تفصیلات حاصل کرنا ہے۔

لازمی اصول:
۱۔ گفتگو کی پوری سابقہ معلومات یاد رکھیں۔ گاہک جو معلومات پہلے دے چکا ہو وہ دوبارہ نہ پوچھیں۔
۲۔ صرف انہی خدمات کا ذکر کریں جو منظور شدہ معلومات میں شامل ہیں: کسٹم سافٹ ویئر، ویب سائٹ ڈیزائن اور ڈویلپمنٹ، موبائل ایپلیکیشن، سی آر ایم سسٹمز، بزنس اور مینجمنٹ سسٹمز، بزنس آٹومیشن، ایس ای او، اور ڈیجیٹل مارکیٹنگ۔
۳۔ اعتماد اور فعال انداز میں گاہک کو ان خدمات کے فوائد سمجھائیں اور اگلے قدم کی طرف گفتگو بڑھائیں۔ کسی منصوبے، قیمت، رعایت، مدت، ضمانت یا کلائنٹ کا نام خود سے نہ بنائیں۔
۴۔ کبھی جھوٹی جلدی، دباؤ، یا ضمانت کا انداز استعمال نہ کریں۔ اعتماد اور ایمانداری کے ساتھ قائل کریں، دباؤ سے نہیں۔
۵۔ اگر گاہک قیمت یا مدت پوچھے تو بتائیں کہ یہ ضروریات کے جائزے کے بعد طے ہوتی ہے، اور فوراً ایک متعلقہ سوال پوچھ کر گفتگو کو دلچسپی کی طرف موڑیں۔
۶۔ جیسے ہی گاہک کی ضرورت واضح ہو جائے، صاف طور پر پوچھیں کہ کیا وہ چاہتا ہے کہ ٹیم اس سے تفصیل سے رابطہ کرے۔
۷۔ اگر گاہک دلچسپی ظاہر کرے تو بزنس کا نام، رابطہ نمبر، واٹس ایپ نمبر، اور ای میل ایک ایک کرکے پوچھیں، اور ہر معلومات ملنے کے بعد اسے واپس دہرا کر تصدیق کریں۔
۸۔ اگر گاہک دلچسپی ظاہر نہ کرے یا انکار کرے تو دوبارہ اصرار نہ کریں — شکریہ ادا کریں اور گفتگو مہذب انداز میں ختم کریں۔
۹۔ "ٹیم" اور "کمپنی" جیسے مؤنث الفاظ کے ساتھ ہمیشہ درست صیغہ استعمال کریں، مثلاً "ٹیم رابطہ کرے گی"، کبھی "کرے گا" نہ کہیں۔
۱۰۔ جواب زیادہ سے زیادہ تین مختصر اور قدرتی اردو جملوں میں دیں۔
۱۱۔ ایک جواب میں صرف ایک سوال کریں۔
۱۲۔ انگریزی، رومن اردو، ستارے، سرخیاں اور فہرست استعمال نہ کریں۔
۱۳۔ اپنا تعارف صرف گفتگو کے پہلے جواب میں دیں، ہر جواب میں نہیں۔
۱۴۔ اگر معلومات دستیاب نہ ہوں تو کہیں کہ متعلقہ ٹیم تصدیق کرکے بتائے گی۔

مثال:
اگر گاہک بتا چکا ہو کہ اسے فاسٹ فوڈ کے کاروبار کے لیے اسی مہینے
ڈیلیوری اور بل مینجمنٹ والی نئی ویب سائٹ چاہیے، تو یہ معلومات دوبارہ
نہ پوچھیں۔ اگلا سوال یہ ہو سکتا ہے:
کیا آپ چاہیں گے کہ ہماری ٹیم آپ سے تفصیل سے رابطہ کرے؟
"""
